In [ ]:
#Consulta em Duas Etapas (Retrieve & Re-rank):

import psycopg2
import numpy as np
import pandas as pd
import requests
from scipy.spatial.distance import cosine # Importando a distância de cosseno
from sentence_transformers import SentenceTransformer
import json
import tkinter as tk
from tkinter import messagebox
import time
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
import matplotlib.patches as patches
import traceback
import re
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
# Listas globais para armazenar os rankings de todas as chamadas
lista_ids_ordem_cosseno = []
lista_ids_reordenados_int = []

# --- CONFIGURAÇÕES ---
# Inicializar modelo para gerar o embedding da consulta
print("Carregando o modelo de embeddings...")
modelo = SentenceTransformer('intfloat/multilingual-e5-small')
print("Modelo carregado.")

# Configurações da API Gemini
API_KEY = "Chave API" 
API_URL = 'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent'

# Configurações do Banco de Dados
DB_HOST = "localhost"
DB_NAME = "postgres" # Nome do banco de dados
DB_USER = "postgres"
DB_PORT = 5432
DB_PASS = "senha" 
DB_TABLE = "texto_avaliacao_ebserh" # Nome da tabela com os dados

# --- FUNÇÕES DE BANCO DE DADOS E LÓGICA DE RAG ---

def buscar_todos_os_dados():
    """Busca todos os dados necessários (id, texto_artigo, vector, texto_pagina) do banco."""
    try:
        conexao = psycopg2.connect(host=DB_HOST, port=DB_PORT, database=DB_NAME, user=DB_USER, password=DB_PASS)
        cursor = conexao.cursor()
        
        # Seleciona todas as colunas que serão usadas no processo
        cursor.execute(f"SELECT id, texto_artigo, vector, texto_pagina, nome_arquivo FROM {DB_TABLE}")
        
        resultados = cursor.fetchall()
        cursor.close()
        conexao.close()
        print(f"Encontrados {len(resultados)} registros no banco de dados.")
        return resultados
    except Exception as e:
        print(f"Erro ao conectar ou buscar dados do banco: {e}")
        return []

def recuperacao_inicial_por_similaridade(query_embedding, dados_banco, k=100):
    """
    Calcula a similaridade de cossenos e retorna os k chunks mais próximos.
    A distância de cosseno é 1 - similaridade de cosseno. Menor distância = maior similaridade.
    """
    # Calcula a distância de cosseno entre a query e todos os embeddings do banco
    # A estrutura de 'dado' agora é (id, texto_artigo, vector, texto_pagina)
    resultados_com_distancia = [
        (dado, cosine(query_embedding, np.array(json.loads(dado[2])).flatten()))
        for dado in dados_banco
    ]
    
    # Ordena pela distância (menor para o maior) e pega os top k
    resultados_ordenados = sorted(resultados_com_distancia, key=lambda x: x[1])[:k]
    
    # Retorna apenas os dados dos chunks, sem a distância
    top_k_chunks = [dado for dado, dist in resultados_ordenados]
    print(f"Recuperação inicial concluída. {len(top_k_chunks)} chunks selecionados.")
    return top_k_chunks
    #print(f"Recuperação inicial concluída. {len(resultados_ordenados)} chunks selecionados a partir de registros válidos.")
    #return resultados_ordenados # Retorna a lista de tuplas (dado, distancia)

def imprimir_rankings_para_comparacao(top_100_chunks, top_100_rrf):
    global lista_ids_ordem_cosseno, lista_ids_reordenados_int

    # 1. Extrai a lista de IDs da busca por cosseno (são inteiros)
    ids_ordem_cosseno = [chunk[0] for chunk in top_100_chunks]
    ids_ordem_rrf = [chunk[0] for chunk in top_100_rrf]
    # 2. Converte os IDs do Gemini (que são strings) para inteiros para uma comparação justa
   # ids_reordenados_int = []
   # for id_str in ids_reordenados:
       # try:
          #  ids_reordenados_int.append(int(id_str))
        #except (ValueError, TypeError):
           # print(f"Aviso: ID '{id_str}' retornado pelo Gemini não é um inteiro e será ignorado.")

    # 3. Armazena os rankings gerados nesta chamada
    lista_ids_ordem_cosseno.append(ids_ordem_cosseno)
    lista_ids_reordenados_int.append(ids_ordem_rrf)

def bm25_rank(top_chunks, query):
    """
    Recebe os 100 chunks recuperados por similaridade de cosseno e realiza reranqueamento com BM25.
    Retorna os mesmos chunks ordenados pela relevância BM25.
    """
    # Extrair os textos dos chunks (ajuste o índice conforme necessário)
    textos = [chunk[1] for chunk in top_chunks]  # Exemplo: chunk[1] é o 'texto_artigo'

    # Tokenização
    corpus_tokenizado = [word_tokenize(doc.lower()) for doc in textos]
    query_tokenizada = word_tokenize(query.lower())

    # Inicializa o modelo BM25
    bm25 = BM25Okapi(corpus_tokenizado)

    # Calcula os scores
    scores = bm25.get_scores(query_tokenizada)

    # Associa cada chunk ao seu score
    chunks_com_score = list(zip(top_chunks, scores))

    # Ordena pelo score decrescente
    rerankeado = sorted(chunks_com_score, key=lambda x: x[1], reverse=True)

    # Retorna só os chunks reranqueados
    return [chunk for chunk, score in rerankeado]

# ======= RRF =======

def rrf_rank(ranking1, ranking2, k=60):
    """
    Aplica Reciprocal Rank Fusion (RRF) nos rankings de chunks.
    Recebe dois rankings (listas de chunks) e retorna uma lista de chunks reranqueada.
    """
    from collections import defaultdict

    # Mapeia IDs para os próprios chunks
    id_para_chunk = {chunk[0]: chunk for chunk in ranking1 + ranking2}

    # Inicializa os scores RRF
    scores_rrf = defaultdict(float)

    # Ranking 1: Similaridade de Cosseno
    for rank, chunk in enumerate(ranking1):
        chunk_id = chunk[0]
        scores_rrf[chunk_id] += 1 / (k + rank)

    # Ranking 2: BM25
    for rank, chunk in enumerate(ranking2):
        chunk_id = chunk[0]
        scores_rrf[chunk_id] += 1 / (k + rank)

    # Ordena os chunks por score RRF
    ids_ordenados = sorted(scores_rrf.items(), key=lambda x: x[1], reverse=True)

    # Retorna a lista de chunks reranqueada
    return [id_para_chunk[chunk_id] for chunk_id, _ in ids_ordenados]

def gerar_resposta_final_com_gemini(query, chunks):
    contexto = ""
    for i, chunk in enumerate(chunks):
        contexto += f"""
Você é um especialista em análise de documentos.

Sua tarefa é responder à pergunta com base nos textos fornecidos.

⚠️ REGRAS IMPORTANTES:
- Gere UMA ÚNICA resposta final
- NÃO responda por documento
- NÃO escreva "Documento 1", "Documento 2", etc.
- NÃO liste múltiplas respostas separadas
- NÃO repita trechos
- Seja objetiva e direta

- Se a resposta NÃO estiver presente nos textos, responda exatamente:
NÃO ENCONTRADO

    Pergunta:
    {query}

    Documentos:
    {contexto}
    """

    payload = {
        "contents": [{"parts": [{"text": prompt}]}]
    }

    try:
        response = requests.post(
            f"{API_URL}?key={API_KEY}",
            headers={'Content-Type': 'application/json'},
            json=payload
        )

        data = response.json()
        resposta = data['candidates'][0]['content']['parts'][0]['text'].strip()

        # 🔥 CONTAGEM DE TOKENS
        tokens_prompt = contar_tokens(prompt)
        tokens_resposta = contar_tokens(resposta)
        total_tokens = tokens_prompt + tokens_resposta

        return {
            "resposta": resposta,
            "tokens_prompt": tokens_prompt,
            "tokens_resposta": tokens_resposta,
            "tokens_total": total_tokens
        }

    except Exception as e:
        print("Erro:", e)
        return {
            "resposta": "ERRO",
            "tokens_prompt": 0,
            "tokens_resposta": 0,
            "tokens_total": 0
        }

def orquestrar_rag_duas_etapas(query):
    try:
        # =========================
        # 1. Buscar dados
        # =========================
        dados_banco = buscar_todos_os_dados()
        if not dados_banco:
            return {
                "pergunta": query,
                "resposta": "ERRO_BANCO",
                "tokens_prompt": 0,
                "tokens_resposta": 0,
                "tokens_total": 0
            }

        # =========================
        # 2. Embedding da query
        # =========================
        embedding_query = modelo.encode(query).tolist()

        # =========================
        # 3. Recuperação inicial
        # =========================
        top_100_chunks = recuperacao_inicial_por_similaridade(
            embedding_query, dados_banco, k=100
        )

        # =========================
        # 4. BM25
        # =========================
        top_100_bm25 = bm25_rank(top_100_chunks, query)

        # =========================
        # 5. RRF
        # =========================
        top_100_rrf_chunks = rrf_rank(top_100_chunks, top_100_bm25, k=60)

        if not top_100_rrf_chunks:
            return {
                "pergunta": query,
                "resposta": "ERRO_RRF",
                "tokens_prompt": 0,
                "tokens_resposta": 0,
                "tokens_total": 0
            }

        # =========================
        # 6. Selecionar TOP 50
        # =========================
        top_50_chunks = top_100_rrf_chunks[:50]

        # =========================
        # 7. Mapas auxiliares
        # =========================
        mapa_id_para_texto = {dado[0]: dado[3] for dado in dados_banco}
        mapa_id_para_nome = {dado[0]: dado[4] for dado in dados_banco}

        # =========================
        # 8. Criar contexto estruturado
        # =========================
        contexto_estruturado = []

        for chunk in top_50_chunks:
            chunk_id = chunk[0]

            if chunk_id in mapa_id_para_texto:
                contexto_estruturado.append({
                    "fonte": mapa_id_para_nome.get(chunk_id, "desconhecido"),
                    "texto": mapa_id_para_texto[chunk_id]
                })

        print(f"Contexto estruturado criado com {len(contexto_estruturado)} chunks")

        # ⚠️ proteção extra
        if not contexto_estruturado:
            return {
                "pergunta": query,
                "resposta": "SEM_CONTEXTO",
                "tokens_prompt": 0,
                "tokens_resposta": 0,
                "tokens_total": 0
            }

        # =========================
        # 9. Chamada ao LLM
        # =========================
        resposta_final = gerar_resposta_final_com_gemini(query, contexto_estruturado)

        # =========================
        # 10. Retorno estruturado
        # =========================
        return {
            "pergunta": query,
            "resposta": resposta_final.get("resposta", "ERRO"),
            "tokens_prompt": resposta_final.get("tokens_prompt", 0),
            "tokens_resposta": resposta_final.get("tokens_resposta", 0),
            "tokens_total": resposta_final.get("tokens_total", 0)
        }

    except Exception as e:
        print("❌ Erro na função orquestrar:")
        traceback.print_exc()

        return {
            "pergunta": query,
            "resposta": "ERRO",
            "tokens_prompt": 0,
            "tokens_resposta": 0,
            "tokens_total": 0
        }
def contar_tokens(texto):
    if not texto:
        return 0
    return int(len(texto) / 4)


  
def buscar_perguntas(limit=193):
    try:
        # --------------------------------------------------
        # 1) Leitura do CSV
        # --------------------------------------------------
        df = pd.read_csv(
            "perguntas_respostas_unificadas.xls",
            encoding="utf-8"
        )

        # --------------------------------------------------
        # 2) Seleciona as primeiras perguntas (ordem do CSV)
        # --------------------------------------------------
        perguntas = df["pergunta"].head(limit).tolist()

        return perguntas

    except Exception as e:
        print("❌ Erro ao buscar perguntas do arquivo CSV:")
        traceback.print_exc()
        return []

     
# --- BLOCO DE EXECUÇÃO PRINCIPAL ---
if __name__ == "__main__":
    perguntas = buscar_perguntas(limit=193)
    resultados = []
    
    for i, pergunta in enumerate(perguntas, 1):
        print(f"\n🔍 Processando pergunta {i}: \"{pergunta}\"\n")

        try:
            resultado = orquestrar_rag_duas_etapas(pergunta)
            resultados.append(resultado)
            print("\n" + "="*100 + f" RESPOSTA {i} " + "="*100)
            print(resultado)
            print("="*192)

        except Exception as e:
            print(f"\n--- ERRO ao processar pergunta {i} ---")
            traceback.print_exc()
            resultados.append({
                "pergunta": pergunta,
                "resposta": "ERRO",
                "tokens_prompt": 0,
                "tokens_resposta": 0,
                "tokens_total": 0
            })
    # Criação dos DataFrames
    df_resultados = pd.DataFrame(resultados)
   
    # Salva resultados em arquivos CSV
    df_resultados.to_csv('resultados1_50chunks_gemini_2_5.csv', index=False)
        
   # print("\n📋 DataFrame - Cosine")
   # print(df_cosine.head())

    print("\n📋 DataFrame - Respostas")
    print(df_resultados.head())
    print("\n✅ Todas as perguntas foram processadas.")
    
    

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mvcun\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Carregando o modelo de embeddings...
Modelo carregado.

🔍 Processando pergunta 1: "Quais formulários são mencionados no documento como disponíveis no site do HE -> Sistemas -> ADS Hospitalar -> Documentos -> USOST – Unidade de Saúde Ocupacional e Segurança do Trabalho, e para quais finalidades são utilizados?"

Encontrados 740 registros no banco de dados.
Recuperação inicial concluída. 100 chunks selecionados.
Contexto estruturado criado com 50 chunks

==================================================================================================== RESPOSTA 1 ====================================================================================================
{'pergunta': 'Quais formulários são mencionados no documento como disponíveis no site do HE -> Sistemas -> ADS Hospitalar -> Documentos -> USOST – Unidade de Saúde Ocupacional e Segurança do Trabalho, e para quais finalidades são utilizados?', 'resposta': 'Os formulários mencionados no documento como disponíveis no site do HE ->

In [ ]:
# Avaliação respostas geradas com BertScore
import pandas as pd
from bert_score import score

# --- BLOCO DE EXECUÇÃO PRINCIPAL ---
if __name__ == "__main__":
    
    # --------------------------------------------------------------
    # 1) Carrega as respostas de referência (gabarito) do CSV
    # --------------------------------------------------------------
    df_gabarito = pd.read_csv("perguntas_respostas_unificadas.xls")

    # Converte tudo para string e remove NaN
    df_gabarito["resposta"] = df_gabarito["resposta"].fillna("").astype(str)

    respostas_referencia = df_gabarito["resposta"].tolist()
    

    # --------------------------------------------------------------
    # 2) Carrega as respostas geradas pelo re-ranker
    # --------------------------------------------------------------
    df_respostas_rff = pd.read_csv(
        "resultados1_50chunks_gemini_2_5.csv"
    )

    # Converte para string
    df_respostas_rff["resposta"] = df_respostas_rff["resposta"].fillna("").astype(str)


    # Verificação de segurança
    if len(df_respostas_rff) != len(respostas_referencia):
        print(f"⚠️ Aviso: número de respostas não bate! {len(df_respostas_rff)} vs {len(respostas_referencia)}")
    
    # Ajustar tamanho mínimo comum
    n = min(len(df_respostas_rff), len(respostas_referencia))

    respostas_geradas = df_respostas_rff["resposta"].iloc[:n].tolist()
    referencias = respostas_referencia[:n]


    # --------------------------------------------------------------
    # 3) Calcula BERTScore
    # --------------------------------------------------------------
    print("Calculando BERTScore...")
    P, R, F1 = score(respostas_geradas, referencias, lang="pt", verbose=True)


    # --------------------------------------------------------------
    # 4) Monta DataFrame com resultados
    # --------------------------------------------------------------
    df_resultados = pd.DataFrame({
        "resposta_gerada": respostas_geradas,
        "resposta_referencia": referencias,
        "precision": P.tolist(),
        "recall": R.tolist(),
        "f1": F1.tolist()
    })


    # --------------------------------------------------------------
    # 5) Salva em CSV
    # --------------------------------------------------------------
    df_resultados.to_csv('resultados1_bertscore_50chunks_Gemini_2_5.csv', index=False)

    print("✅ Arquivo 'resultados1_bertscore_50chunks_Gemini_2_5.csv' salvo com sucesso.")
    print(f"\n✅ Média F1: {F1.mean().item():.4f}")


Calculando BERTScore...
calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 103.41 seconds, 1.86 sentences/sec
✅ Arquivo 'resultados1_bertscore_50chunks_Gemini_2_5.csv' salvo com sucesso.

✅ Média F1: 0.7987
